# Chess Rating Systems: Building and Comparing Elo & Glicko-2

**Author:** Atilla Ahmed.
**Data Sources:** Lichess Game Database (Kaggle) · FIDE Official Rating Lists

## Table of Content

1. Introduction
2. Data Loading & Initial Inspection
3. Data Cleaning & Preprocessing
4. Exploratory Data Analysis
5. Mathematical Framework
6. Implementation
7. Hypothesis Testing
8. Conclusion
9. Self-Assessment

---

## 1. Introduction

### 1.1 Motivation

I play chess online and I've always been curious about what the rating number actually means mathematically. The **Elo system**, introduced in the 1960s by Arpad Elo and adopted by FIDE in 1970, uses a logistic function to estimate the probability that one player beats another based on their rating difference. It's simple and widely used, but it has a weakness it doesn't track how *certain* the system is about a player's rating.

**Glicko-2** (Glickman, 2001) solves this by adding a rating deviation (uncertainty) and a volatility parameter per player. Lichess uses Glicko-2; FIDE still uses Elo. In this project I derive both systems from scratch, implement them in Python, and test them on real data.

### 1.2 Problem Statement & Objectives

> **How well do Elo and Glicko-2 predict chess outcomes, and what do online (Lichess) and over-the-board (FIDE) data reveal about how ratings behave in practice?**

Specific questions I want to answer:

1. How well-calibrated is the Elo prediction? (If the model says 70% win chance, does white win ~70% of the time?)
2. Is white's first-move advantage statistically significant, and does it vary by rating level?
3. Do certain openings produce significantly different win rates?
4. How do Lichess and FIDE rating distributions compare structurally?
5. Is there a gender gap in FIDE ratings, and can participation rates explain it?
6. Can a logistic regression model that includes color, opening, and time control outperform pure Elo?

Each of these is formalized as a hypothesis test or model comparison in Sections 6–7.

The project has three goals: derive the math behind Elo and Glicko-2 with full formulas, implement both from scratch (no pre-built rating libraries), and run statistical analysis on real data from two independent sources.

### 1.3 Data Sources

**Lichess (Kaggle)** - 20,000+ online games in CSV format. Each row is a game with both players' ratings, the result, opening name/ECO code, number of moves, and time control. Licensed CC0.

**FIDE (ratings.fide.com)** - official rating list in fixed-width TXT. Each row is a player with Standard/Rapid/Blitz ratings, title, federation, gender, and birth year.

The two datasets can't be merged at the player level (no shared ID), but I compare them at the aggregate level, distribution shapes, population parameters, and how the Elo formula performs in each context.

### 1.4 Prior Work

Elo described his system in *The Rating of Chessplayers, Past and Present* (1978). Glickman published Glicko-2 in 2001 — his paper *"Example of the Glicko-2 System"* is the main reference I follow for the implementation. There's also ML-based approaches (logistic regression, gradient boosting) that can outperform Elo in raw accuracy, but I'm more interested in understanding the mathematical structure than maximizing a metric.

## 2. Data Loading and Initial Inspection
This section Loads both datasets, check their structure, and flags anything that need attention before the real analysis. The Lichess data comes as a standard CSV with one row per game. The FIDE data is a fixed-width text file a snapshot of every registered player with their current ratings.

### 2.0 Setup and Imports

In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, shapiro, binomtest

### 2.1 Loading the Lichess Dataset
The Lichess data is a CSV from Kaggle containing ~20,000 games played.


In [2]:
lichess = pd.read_csv('games.csv')
# Let's look the shape
print(f'Shape: {lichess.shape[0]:,} rows x {lichess.shape[1]:,} columns ')
print()

# Columns types and non-null counts
lichess.info()

Shape: 20,058 rows x 16 columns 

<class 'pandas.DataFrame'>
RangeIndex: 20058 entries, 0 to 20057
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              20058 non-null  str    
 1   rated           20058 non-null  bool   
 2   created_at      20058 non-null  float64
 3   last_move_at    20058 non-null  float64
 4   turns           20058 non-null  int64  
 5   victory_status  20058 non-null  str    
 6   winner          20058 non-null  str    
 7   increment_code  20058 non-null  str    
 8   white_id        20058 non-null  str    
 9   white_rating    20058 non-null  int64  
 10  black_id        20058 non-null  str    
 11  black_rating    20058 non-null  int64  
 12  moves           20058 non-null  str    
 13  opening_eco     20058 non-null  str    
 14  opening_name    20058 non-null  str    
 15  opening_ply     20058 non-null  int64  
dtypes: bool(1), float64(2), int64(4), str(9)
memory usage

In [3]:
# Preview first 5 rows
lichess.head()

,id,rated,created_at,last_move_at,turns,victory_status,winner,increment_code,white_id,white_rating,black_id,black_rating,moves,opening_eco,opening_name,opening_ply
0,TZJHLljE,False,1.504210e+12,1.504210e+12,13,outoftime,white,15+2,bourgris,1500,a-00,1191,d4 d5 c4 c6 cxd5 e6 dxe6 fxe6 Nf3 Bb4+ Nc3 Ba5...,D10,Slav Defense: Exchange Variation,5
1,l1NXvwaE,True,1.504130e+12,1.504130e+12,16,resign,black,5+10,a-00,1322,skinnerua,1261,d4 Nc6 e4 e5 f4 f6 dxe5 fxe5 fxe5 Nxe5 Qd4 Nc6...,B00,Nimzowitsch Defense: Kennedy Variation,4
2,mIICvQHh,True,1.504130e+12,1.504130e+12,61,mate,white,5+10,ischia,1496,a-00,1500,e4 e5 d3 d6 Be3 c6 Be2 b5 Nd2 a5 a4 c5 axb5 Nc...,C20,King's Pawn Game: Leonardis Variation,3
3,kWKvrqYL,True,1.504110e+12,1.504110e+12,61,mate,white,20+0,daniamurashov,1439,adivanov2009,1454,d4 d5 Nf3 Bf5 Nc3 Nf6 Bf4 Ng4 e3 Nc6 Be2 Qd7 O...,D02,Queen's Pawn Game: Zukertort Variation,3
4,9tXo1AUZ,True,1.504030e+12,1.504030e+12,95,mate,white,30+3,nik221107,1523,adivanov2009,1469,e4 e5 Nf3 d6 d4 Nc6 d5 Nb4 a3 Na6 Nc3 Be7 b4 N...,C41,Philidor Defense,5


In [4]:
# Statistics for Numerical Columns
lichess.describe().T

,count,mean,std,min,25%,50%,75%,max
created_at,20058.0,1.483617e+12,2.850151e+10,1.376772e+12,1.477548e+12,1.496010e+12,1.503170e+12,1.504493e+12
last_move_at,20058.0,1.483618e+12,2.850140e+10,1.376772e+12,1.477548e+12,1.496010e+12,1.503170e+12,1.504494e+12
turns,20058.0,6.046600e+01,3.357058e+01,1.000000e+00,3.700000e+01,5.500000e+01,7.900000e+01,3.490000e+02
white_rating,20058.0,1.596632e+03,2.912534e+02,7.840000e+02,1.398000e+03,1.567000e+03,1.793000e+03,2.700000e+03
black_rating,20058.0,1.588832e+03,2.910361e+02,7.890000e+02,1.391000e+03,1.562000e+03,1.784000e+03,2.723000e+03
opening_ply,20058.0,4.816981e+00,2.797152e+00,1.000000e+00,3.000000e+00,4.000000e+00,6.000000e+00,2.800000e+01


Check for missing values the '.info()' above showed all columns as non-null, but i will check to confirm is everything is okay.

In [5]:
missing_lichess = lichess.isnull().sum()
missing_pct = (missing_lichess / len(lichess)) * 100 # as percentage of total

# Now display columns with missing values (if any)
missing_df = pd.DataFrame({
    'Missing': missing_lichess,
    'Percentage': missing_pct
})
missing_df[missing_df['Missing'] > 0] # Filtering to show only problematic columns
# If this table is empty - no missing data at all

,Missing,Percentage


Unique values for key categorical columns, this tells me how many distinct players, openings, and result types exists:

In [6]:
cat_cols = ['winner', 'victory_status', 'increment_code', 'opening_eco', 'opening_name']
for col in cat_cols:
    n_unique = lichess[col].nunique() # count distinct values
    top_val = lichess[col].value_counts().index[0] # most frequent value
    top_pct = lichess[col].value_counts(normalize=True).iloc[0] * 100
    print(f'{col:20s} → {n_unique:>5,} unique | most common: "{top_val}" ({top_pct:.1f}%)')

winner               →     3 unique | most common: "white" (49.9%)
victory_status       →     4 unique | most common: "resign" (55.6%)
increment_code       →   400 unique | most common: "10+0" (38.5%)
opening_eco          →   365 unique | most common: "A00" (5.0%)
opening_name         → 1,477 unique | most common: "Van't Kruijs Opening" (1.8%)


### 2.2 Loading the FIDE Dataset
The FIDE data is a rating list snapshot from ratings.fide.com with ~1.8 million registered players. Each row is one player with their Standard, Rapid, and Blitz Elo ratings, plus metadata like title, federation, gender, and birth year. The column names use FIDE's standard abbreviations — I'll rename them for clarity.

In [7]:
fide = pd.read_fwf('players_list_foa.txt')
print(f'Shape: {fide.shape[0]:,} rows x {fide.shape[1]:,} columns ')
print()

print('Columns:', list(fide.columns))
print()

print(fide.dtypes)


Shape: 1,816,699 rows x 19 columns 

Columns: ['ID Number', 'Name', 'Fed', 'Sex', 'Tit', 'WTit', 'OTit', 'FOA', 'SRtng', 'SGm', 'SK', 'RRtng', 'RGm', 'Rk', 'BRtng', 'BGm', 'BK', 'B-day', 'Flag']

ID Number    int64
Name           str
Fed            str
Sex            str
Tit            str
WTit           str
OTit           str
FOA            str
SRtng        int64
SGm          int64
SK           int64
RRtng        int64
RGm          int64
Rk           int64
BRtng        int64
BGm          int64
BK           int64
B-day        int64
Flag           str
dtype: object


FIDE uses cryptic column abbreviations. Here's what each one means, i'll rename some of them later for readability:

| Column | Meaning |
|--------|---------|
| `SRtng` / `SGm` / `SK` | Standard rating, games, K-factor |
| `RRtng` / `RGm` / `Rk` | Rapid rating, games, K-factor |
| `BRtng` / `BGm` / `BK` | Blitz rating, games, K-factor |
| `Tit` | FIDE title (GM, IM, FM, CM, WGM, ...) |
| `WTit` | Women's title |
| `Fed` | Federation (country code, e.g. BUL, USA, RUS) |
| `B-day` | Birth year |
| `Flag` | Activity flag (i = inactive, w = woman-inactive, etc.) |

In [8]:
# First 5 rows:
fide.head()

,ID Number,Name,Fed,Sex,Tit,WTit,OTit,FOA,SRtng,SGm,SK,RRtng,RGm,Rk,BRtng,BGm,BK,B-day,Flag
0,564033791,564033783,IND,M,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,2017,NaN
1,40129322,9147028,NED,M,NaN,NaN,NaN,NaN,0,0,0,0,0,0,1530,0,20,1995,NaN
2,10292519,"A A M Imtiaz, Chowdhury",BAN,M,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,1975,NaN
3,10688862,"A Abdel Maabod, Hoda",EGY,F,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,2009,w
4,577017641,A Adhisiva,IND,M,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,2015,NaN


In [9]:
# Descriptive Statistics
fide.describe().T

,count,mean,std,min,25%,50%,75%,max
ID Number,1816699.0,1.105937e+08,1.948319e+08,100013.0,6559127.0,24132403.0,54196802.0,706088288.0
SRtng,1816699.0,5.198892e+02,8.073976e+02,0.0,0.0,0.0,1510.0,2840.0
SGm,1816699.0,1.649266e-01,1.030813e+00,0.0,0.0,0.0,0.0,36.0
SK,1816699.0,9.367054e+00,1.539785e+01,0.0,0.0,0.0,20.0,40.0
RRtng,1816699.0,4.050264e+02,7.268124e+02,0.0,0.0,0.0,0.0,2832.0
RGm,1816699.0,1.305549e-01,1.194110e+00,0.0,0.0,0.0,0.0,311.0
Rk,1816699.0,7.141272e+00,1.362389e+01,0.0,0.0,0.0,0.0,40.0
BRtng,1816699.0,2.782552e+02,6.412555e+02,0.0,0.0,0.0,0.0,2869.0
BGm,1816699.0,9.610563e-02,1.188960e+00,0.0,0.0,0.0,0.0,261.0
BK,1816699.0,4.077000e+00,9.989860e+00,0.0,0.0,0.0,0.0,40.0


Missing values in FIDE data, many players don't have Rapid or Blitz ratings, and titles are only held by a small fraction of registered players:

In [10]:
# Missing values per column
missing_fide = fide.isnull().sum()                     # count NaN per column
missing_fide_pct = (missing_fide / len(fide) * 100)    # as percentage

missing_fide_df = pd.DataFrame({
    'Missing': missing_fide,
    'Percent': missing_fide_pct
}).sort_values('Percent', ascending=False)              # sort by severity

missing_fide_df[missing_fide_df['Missing'] > 0]        # show only columns with gaps

,Missing,Percent
WTit,1811671,99.723234
OTit,1811165,99.695382
FOA,1801280,99.151263
Tit,1792550,98.670721
Flag,1244083,68.480414


In [11]:
#  Check for "unrated" players (rating = 0 means no official rating)
for col in ['SRtng', 'RRtng', 'BRtng']:                # the three rating columns
    n_zero = (fide[col] == 0).sum()                      # count players with rating = 0
    pct_zero = n_zero / len(fide) * 100                  # as percentage
    print(f'{col}: {n_zero:>9,} zeros ({pct_zero:.1f}%)')

SRtng: 1,275,497 zeros (70.2%)
RRtng: 1,379,776 zeros (75.9%)
BRtng: 1,524,341 zeros (83.9%)


### 2.3 First Impressions

**Lichess:** 20,058 games, zero missing values across all 16 columns. Ratings range from 784 to 2723 (mean ~1,597 for white, ~1,589 for black), which covers everything from absolute beginners to strong masters. The average game lasts about 60 moves, with a maximum of 349 some real marathon games in there. White wins 49.9% of the time, resign is the most common result (55.6%), and there are 1,477 distinct opening names. No cleaning needed for this dataset.

**FIDE:** 1.8 million registered players, but the majority are "ghost entries" players registered with a federation who have never earned an official rating. Standard rating is zero for 70.2% of players, Rapid for 75.9%, and Blitz for 83.9%. After filtering out the zeros, I'll be working with roughly 541K players for Standard, 437K for Rapid, and 292K for Blitz still very large samples. Title columns are mostly empty (98.7% for `Tit`) because only a tiny fraction of chess players ever earn a FIDE title. The `Flag` column is NaN for 68.5% of entries, which likely means "active" the flag values mark inactive or special-status players.

Both datasets loaded successfully. The Lichess data is ready to use as-is. The FIDE data needs filtering (remove zero-rated entries) and some column renaming for readability before I can do anything meaningful with it.

## 3. Data Cleaning & Preprocessing
The Lichess dataset came in surprisingly clean: no missing values, sensible dtypes, and nothing that looks like a data entry error. The FIDE data is a different story. Most of the 1.8 million rows are 'ghost entries' - players registered with a federation who never got an official rating. I need to filter those out, handle the 'Flag' column, and do some renaming so the column names actually make sense to someone reading this.

### 3.1 Lichess Validation & feature engineering
There's nothing to clean here, but I do want to verify the zero-missing claim from Section 2

In [12]:
# Verify no missing values
print('Missing values per column:')
print(lichess.isnull().sum())
print(f'\nDuplicate rows: {lichess.duplicated().sum()}')

Missing values per column:
id                0
rated             0
created_at        0
last_move_at      0
turns             0
victory_status    0
winner            0
increment_code    0
white_id          0
white_rating      0
black_id          0
black_rating      0
moves             0
opening_eco       0
opening_name      0
opening_ply       0
dtype: int64

Duplicate rows: 429


In [13]:
# Drop duplicates by game ID only
# Same ID = same game exported twice/thrice. Different ID = different game.
print(f'Before: {len(lichess):,} rows')
lichess = lichess.drop_duplicates(subset='id', keep='first')  # keep first occurrence of each game ID
lichess = lichess.reset_index(drop=True)                       # clean index
print(f'After:  {len(lichess):,} rows')
print(f'Dropped: {20058 - len(lichess):,} duplicate game entries')

Before: 20,058 rows
After:  19,113 rows
Dropped: 945 duplicate game entries


945 rows dropped - all cases where the same game ID appeared more than once. The remaining 19,113 rows are unique games. Now I'll create derived columns I need for the Elo analysis.

In [14]:
#Rating difference: white minus black
#This is the main predictor for the Elo expected score formula.
#Positive = white is stronger, negative = black is stronger.
lichess['rating_diff'] = lichess['white_rating'] - lichess['black_rating']
print(f'rating_diff range: [{lichess["rating_diff"].min()}, {lichess["rating_diff"].max()}]')
print(f'Mean: {lichess["rating_diff"].mean():.1f}')
print(f'Median: {lichess["rating_diff"].median():.0f}')

rating_diff range: [-1605, 1499]
Mean: 7.3
Median: 3


In [15]:
# Numeric outcome from white's perspective
# Elo needs a numeric score: 1 for win, 0.5 for draw, 0 for loss.
# The 'winner' column has values: 'white', 'black', 'draw'.
outcome_map = {'white': 1.0, 'draw': 0.5, 'black': 0.0}  # map strings to floats
lichess['white_score'] = lichess['winner'].map(outcome_map)  # apply the mapping

# Verify no unmapped values crept in
print(f'Any NaN in white_score: {lichess["white_score"].isna().sum()}')
print(f'\nwhite_score distribution:')
print(lichess['white_score'].value_counts().sort_index())

Any NaN in white_score: 0

white_score distribution:
white_score
0.0    8680
0.5     888
1.0    9545
Name: count, dtype: int64


In [16]:
# Average rating of both players
# Useful for grouping games by skill level later.
lichess['avg_rating'] = (lichess['white_rating'] + lichess['black_rating']) / 2

# Rating band (every 200 points)
# Bins like 800-1000, 1000-1200, etc. for grouped analysis.
lichess['rating_band'] = (lichess['avg_rating'] // 200 * 200).astype(int)

print('Rating band distribution:')
print(lichess['rating_band'].value_counts().sort_index())

Rating band distribution:
rating_band
800       87
1000    1023
1200    3423
1400    5845
1600    4467
1800    2848
2000    1165
2200     244
2400      11
Name: count, dtype: int64
